# Week 5 & 6 Deliverables   

## 1. Download Data

### Samples
- [Short-read Illumina (**interleaved** paired-end FASTQ)](https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/illumina.fq.bz2)
- [Long-read PacBio](https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/pacbio.fq.bz2)

### Reference Genome
The hg38 (or GRCh38) version of the human genome, focusing on the chromosome that contains these genes ([chromosome 10](https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr10.fa.gz)): 
- [CYP2C8](https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr10%3A95036772%2D95069497&hgsid=3290464905_o8n5jQXlACoTC3asJu2IgMaUIkFs) (regulates many drugs, including anticancer, diabetes and blood pressure drugs)
- [CYP2C9](https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr10%3A94938658%2D94990091&hgsid=3290512893_ZsVBPQDxXLev3NROCdUXxmaa0Mr2) (regulates many common drugs, including warfarin / Coumadin and NSAIDs such as Advil)
- [CYP2C19](https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr10%3A94762681%2D94855547&hgsid=3290512893_ZsVBPQDxXLev3NROCdUXxmaa0Mr2) (regulates… yup, many common drugs, including antiplatelet drugs, antidepressants and anti-epileptic drugs).            

|Genes | CYP2C8 | CYP2C9 | CYP2C19 |
| --- | --- | --- | ---|
| Genomic sequence | chr10:95036772-95069497 | chr10:94938658-94990091 | chr10:94762681-94855547 |
| Strand | - | + | + | 
| Genomic size | 32726 | 51434 | 92867 |

In [ ]:
!mkdir -p data

# SAMPLES
# Download Illumina and PacBio data
!wget -P data/ https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/illumina.fq.bz2
!wget -P data/ https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/pacbio.fq.bz2
!bunzip2 data/*.bz2

In [ ]:
# REFERENCE GENOME 
# chr10 containing CYP2C genes
# Downloading important genes 

from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import requests

# Output FASTA file
output_file = "data/reference_genome.fa"

# Gene coordinates (hg38, UCSC)
GENE_INFO = {
    "CYP2C19": {"chr": "chr10", "start": 94762681, "end": 94855547, "strand": "+"},
    "CYP2C9":  {"chr": "chr10", "start": 94938658, "end": 94990091, "strand": "+"},
    "CYP2C8":  {"chr": "chr10", "start": 95036772, "end": 95069497, "strand": "-"},
}

# UCSC FASTA API template
ucsc_fasta_url = "https://api.genome.ucsc.edu/getData/sequence?genome=hg38;chrom={chr};start={start};end={end}"

records = []

for gene, info in GENE_INFO.items():
    url = ucsc_fasta_url.format(chr=info["chr"], start=info["start"]-1, end=info["end"])
    r = requests.get(url)
    r.raise_for_status()
    seq = r.json()["dna"]

    # Create SeqRecord
    record = SeqRecord(
        Seq(seq),
        id=gene,
        description=f"{info['chr']}:{info['start']}-{info['end']} ({info['strand']})"
    )
    records.append(record)

# Write all genes to a single FASTA
with open(output_file, "w") as f:
    SeqIO.write(records, f, "fasta")

## 2. Align Samples to Reference Genome

**Short-read Illumina (interleaved paired-end FASTQ)**  
- -x: applies multiple options at the same time 
- sr: short read alignment without slicing

**Long-read PacBio**  
- map-hifi: align PacBio high-fidelity reads to a reference genome


In [ ]:
# minimap index
!minimap2 -d data/reference_genome.mmi data/reference_genome.fa 

# Short read Illumina 
!minimap2 -ax sr data/reference_genome.mmi data/illumina.fq > data/illumina.sam 

# Long read BioPac
!minimap2 -ax map-hifi data/reference_genome.mmi data/pacbio.fq > data/pacbio.sam

In [ ]:
# Convert SAM to sorted BAM
!samtools view -bS data/illumina.sam | samtools sort -o data/illumina.bam
!samtools view -bS data/pacbio.sam | samtools sort -o data/pacbio.bam

# Index BAM for random access
!samtools index -b data/illumina.bam
!samtools index -b data/pacbio.bam

## 3. Variant Calling 
`bcftools mpileup -Ou -f data/reference_genome.fa data/illumina.bam | bcftools call -mv --ploidy 2 -Oz -o data/illumina.vcf.gz`
- mpileup part generates genotype likelihoods at each genomic position with coverage
- call part makes the actual calls to interpret above likelihoods and detect variants 
- -m switch tells the program to use the default calling method (multiallelic caller)
- -v option asks to output only variant sites
- --ploidy 2 options assume that the genome is diploid 
- -O option selects the output format -z selects the vcf.gz format 
- -o output file name  
- Do not waste computer’s time by making mpileup convert from the internal binary representation (BCF) to text (VCF), only to be immediately converted back to binary representation by call. Instead, use -Ou to work with uncompressed BCF output

`!bcftools norm -f data/reference_genome.fa -m -any -Oz -o data/illumina.norm.splitted.vcf.gz data/illumina.vcf.gz`
- -m -any options splits multialleleic into one record per ALT 
- -f option uses the reference genome for allele normalization 

In [ ]:
# Index reference genome
!samtools faidx data/reference_genome.fa

In [ ]:
# Call variants 
!bcftools mpileup -Ou -f data/reference_genome.fa data/illumina.bam | bcftools call -mv --ploidy 2 -Oz -o data/illumina.vcf.gz
!bcftools norm -f data/reference_genome.fa -m -any -Oz -o data/illumina.norm.splitted.vcf.gz data/illumina.vcf.gz
!bcftools index data/illumina.norm.splitted.vcf.gz

!bcftools mpileup -Ou -f data/reference_genome.fa data/pacbio.bam | bcftools call -mv --ploidy 2 -Oz -o data/pacbio.vcf.gz
!bcftools norm -f data/reference_genome.fa -m -any -Oz -o data/pacbio.norm.splitted.vcf.gz data/pacbio.vcf.gz
!bcftools index data/pacbio.norm.splitted.vcf.gz

!bcftools convert -O v data/illumina.norm.splitted.vcf.gz > data/illumina.norm.splitted.vcf
!bcftools convert -O v data/pacbio.norm.splitted.vcf.gz > data/pacbio.norm.splitted.vcf

## 4. Phase Variant VCFs 

In [ ]:
!extractHAIRS --bam data/illumina.bam --VCF data/illumina.norm.splitted.vcf --out data/illumina.fragments --ref data/reference_genome.fa 
!HAPCUT2 --fragments data/illumina.fragments --VCF data/illumina.norm.splitted.vcf --output data/illumina.hapcut

!extractHAIRS --pacbio 1 --bam data/pacbio.bam --VCF data/pacbio.norm.splitted.vcf --out data/pacbio.fragments --ref data/reference_genome.fa 
!HAPCUT2 --fragments data/pacbio.fragments --VCF data/pacbio.norm.splitted.vcf --output data/pacbio.hapcut

## 5. Variant Analysis
- Now you should have two phased VCF files (one for each sequencing technology). Compare these VCFs. 
    - How many variants are shared between the VCFs? How many are not?
- Select 2-3 variants that are not common (if any) and check which technology supports this variant. Open both BAM files in IGV and take a screenshot of each problematic discordant location. What can you deduce from these screenshots—are these variants sequencing-related artifacts or are they indeed true variants? Do this analysis for every gene.
- IGV screenshots can also be automated (it is a bit tricky, though—ask your LLM for help). You can opt out of doing this, but you will lose half a point.

- Expected output: Jupyter cell(s) with IGV screenshots and a discussion.

In [ ]:
import pysam 

# Load phased VCFs
illumina_path = "data/illumina.hapcut.phased.VCF"
pacbio_path = "data/pacbio.hapcut.phased.VCF"

illumina_vcf = pysam.VariantFile(illumina_path)
pacbio_vcf = pysam.VariantFile(pacbio_path)

illumina_variants = set()
pacbio_variants = set()

for rec in illumina_vcf.fetch():
    for alt in rec.alts:
        illumina_variants.add((rec.chrom, rec.pos, rec.ref, alt))

for rec in pacbio_vcf.fetch():
    for alt in rec.alts:
        pacbio_variants.add((rec.chrom, rec.pos, rec.ref, alt))

# Compare
shared_variants = illumina_variants & pacbio_variants
illumina_only = illumina_variants - pacbio_variants
pacbio_only = pacbio_variants - illumina_variants

print(f"Shared variants: {len(shared_variants)}")
print(f"Unique to Illumina: {len(illumina_only)}")
print(f"Unique to PacBio:  {len(pacbio_only)}\n")

In [ ]:
discordant_variants = list(illumina_only)[:2] + list(pacbio_only)[:2]
print("Example discordant variants:")
for chrom, pos, ref, alt in discordant_variants:
    print(f"{chrom}:{pos} {ref}->{alt}")

In [ ]:
# Pick 3 variants from Illumina-only and PacBio-only
sample_illumina = list(illumina_only)[:3]
sample_pacbio = list(pacbio_only)[:3]

print("Sample Illumina-only variants:", sample_illumina)
print("Sample PacBio-only variants:", sample_pacbio)

In [ ]:
import subprocess

# Paths
bam_files = {
    "illumina": "data/illumina.bam",
    "pacbio": "data/pacbio.bam"
}
reference = "data/reference_genome.fa"

# Example variant
chrom, pos, ref, alt = sample_illumina[0]
start, end = pos-50, pos+50  # ±50bp window

# IGV batch script content
batch_script = f"""
new
genome {reference}
load {bam_files['illumina']}
goto {chrom}:{start}-{end}
snapshot data/illumina_variant1.png
exit
"""

with open("data/igv_batch.txt", "w") as f:
    f.write(batch_script)

# Run IGV in batch mode
!igv.sh -b data/igv_batch.txt

In [ ]:
# Parse VCFs
from cyvcf2 import VCF
import pandas as pd

# Load phased VCFs
illumina_path = "data/illumina.hapcut.phased.VCF"
pacbio_path = "data/pacbio.hapcut.phased.VCF"

def read_variants(path):
    variants = {}
    vcf = VCF(path)
    for v in vcf:
        key = (v.CHROM, v.POS, v.REF, tuple(v.ALT))

        # String representation of genotypes
        gt = v.gt_bases[0] if v.gt_bases is not None else None

        # Depth value 
        dp = v.INFO.get("DP", None)

        pq = v.format("PQ")[0][0] if "PQ" in v.FORMAT else None

        variants[key] = {
            "CHROM": v.CHROM,
            "POS": v.POS,
            "REF": v.REF,
            "ALT": ",".join(v.ALT),
            "QUAL": v.QUAL,
            "DP": dp,
            "GT": gt,
            "PQ": pq
        }
    return variants

illumina_var = read_variants(illumina_path)
pacbio_var   = read_variants(pacbio_path)

illumina_var_key = set(illumina_var.keys())
pacbio_var_key   = set(pacbio_var.keys())

shared_keys   = illumina_var_key & pacbio_var_key
illumina_only = illumina_var_key - pacbio_var_key
pacbio_only   = pacbio_var_key - illumina_var_key

print(f"Shared variants: {len(shared_keys)}")
print(f"Unique to Illumina: {len(illumina_only)}")
print(f"Unique to PacBio:  {len(pacbio_only)}\n")

for key in list(illumina_only)[:3]:
    v = illumina_var[key]
    print(f"[Illumina-only] {v['CHROM']}:{v['POS']} {v['REF']}->{v['ALT']}  GT={v['GT']}  DP={v['DP']}  PQ={v['PQ']}")

for key in list(pacbio_only)[:3]:
    v = pacbio_var[key]
    print(f"[PacBio-only]  {v['CHROM']}:{v['POS']} {v['REF']}->{v['ALT']}  GT={v['GT']}  DP={v['DP']}  PQ={v['PQ']}")

## 6. Star-Allele Calls
- Can you figure out the star-allele for each gene of interest? The star-allele database can be found in PharmVar; see this for CYP2C19. Your answer should be something like CYP2C19*12 because X, Y and Z. This step does not have to be automated, but should be at least explained in the notebook.
    - Hint: use phased data!

- Expected output: Jupyter cell(s) with discussion (and code, if you want to do it that way).